In [ ]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data import SliceConditionDataset
from inference import predict
from models import CustomVAE, DDPM, SliceConditionedTimeUNet, TimeUNet

DATA_DIR = ROOT / "data" / "images"
VAE_CKPT = ROOT / "microlad-anode" / "vae_anode.pth"
UNET_CKPT = ROOT / "microlad-anode" / "unet_anode.pth"
PATCH_SIZE = 64
AXIS = "z"
AXIS_ID = 0
SLICE_INDEX = 12
TIMESTEPS = 3
SDS_STEPS = 1
SEED = 0


In [ ]:
torch.manual_seed(SEED)
dataset = SliceConditionDataset(DATA_DIR, patch_size=PATCH_SIZE, axis=AXIS, slice_index=SLICE_INDEX, seed=SEED)
batch = next(iter(DataLoader(dataset, batch_size=1, shuffle=False)))
condition = batch["condition"]
condition.shape


In [ ]:
vae = CustomVAE(latent_ch=4)
vae.load_state_dict(torch.load(VAE_CKPT, map_location="cpu")["vae"])
vae.eval()

unet = SliceConditionedTimeUNet(latent_ch=4, base_ch=16, time_dim=16, max_slices=64)
unet.eval()

sds_unet = TimeUNet(latent_ch=4, base_ch=128, time_dim=64)
sds_unet.load_state_dict(torch.load(UNET_CKPT, map_location="cpu"))
sds_unet.eval()

ddpm = DDPM(timesteps=TIMESTEPS)


In [ ]:
result = predict(
    vae=vae,
    unet=unet,
    ddpm=ddpm,
    condition=condition,
    axis=AXIS_ID,
    slice_index=SLICE_INDEX,
    volume_shape=(4, 16, 16, 16),
    sds_steps=SDS_STEPS,
    sds_unet=sds_unet,
    t_min=1,
    t_max=TIMESTEPS,
    use_condition_tpc=True,
    condition_tpc_weight=1.0,
    lock_condition_slice=True,
)

locked_error = (result["volume"][SLICE_INDEX] - condition.squeeze(0)).abs().max()
result["volume_z"].shape, result["volume"].shape, len(result["sds_history"]), float(locked_error)
